# MuseTalk Test — Armageddon Lipsync

End-to-end test: YouTube Short → ElevenLabs dub → MuseTalk lipsync

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# ═══════════════════════════════════════════
# CONFIG — set these before running
# ═══════════════════════════════════════════

YOUTUBE_URL = "https://www.youtube.com/shorts/rPZ89xFA8ww"
TARGET_LANG = "es"  # Spanish (change to: fr, de, pt, ja, ko, etc.)
ELEVENLABS_API_KEY = ""  # paste your key here

In [ ]:
# ═══════════════════════════════════════════
# STEP 1: Install everything
# ═══════════════════════════════════════════
# Colab is Python 3.12 — MMLab's toolchain is broken on it.
# We fix setuptools, skip chumpy, and install from pre-built wheels.

import os

# Fix setuptools for Python 3.12 (restores pkg_resources.ImpImporter)
!pip install -q "setuptools<71"

!pip install -q yt-dlp httpx

# Clone MuseTalk
if not os.path.isdir("MuseTalk"):
    !git clone https://github.com/TMElyralab/MuseTalk.git

%cd MuseTalk

# Install MuseTalk requirements (ignore errors from version conflicts)
!pip install -q -r requirements.txt 2>&1 | tail -5

# Pin diffusers to avoid Flax deprecation noise
!pip install -q "diffusers<1.0.0"

# MMLab stack — install from pre-built wheels, skip broken deps
!pip install -q openmim

# mmengine and mmcv need to match PyTorch/CUDA versions
!mim install -q mmengine
!mim install -q mmcv==2.0.1

# mmdet and mmpose — skip chumpy (broken on 3.12, not needed for inference)
!pip install -q mmdet==3.1.0 --no-deps
!pip install -q mmpose==1.1.0 --no-deps

# Install the deps mmdet/mmpose actually need (without chumpy)
!pip install -q xtcocotools scipy munkres json_tricks

# Verify
import importlib
print("\nMMLab package check:")
for pkg in ["mmengine", "mmcv", "mmdet", "mmpose"]:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, "__version__", "installed")
        print(f"  {pkg}: {v}")
    except ImportError as e:
        print(f"  {pkg}: FAILED — {e}")

In [ ]:
# ═══════════════════════════════════════════
# STEP 2: Download MuseTalk model weights
# ═══════════════════════════════════════════

!bash download_weights.sh

# Verify
for f in ["models/musetalkV15/unet.pth", "models/musetalkV15/musetalk.json"]:
    status = 'OK' if os.path.isfile(f) else 'MISSING'
    print(f"  [{status}] {f}")

In [ ]:
# ═══════════════════════════════════════════
# STEP 3: Download the YouTube Short
# ═══════════════════════════════════════════

import subprocess
import json

os.makedirs("test_data", exist_ok=True)
VIDEO_PATH = "test_data/source.mp4"

subprocess.run(
    [
        "yt-dlp",
        "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4",
        "--merge-output-format", "mp4",
        "-o", VIDEO_PATH,
        YOUTUBE_URL,
    ],
    check=True,
)

size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
print(f"Downloaded: {VIDEO_PATH} ({size_mb:.1f} MB)")

# Show video info
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", "-show_streams", VIDEO_PATH],
    capture_output=True, text=True,
)
d = json.loads(probe.stdout)
dur = float(d["format"]["duration"])
for s in d["streams"]:
    if s["codec_type"] == "video":
        fps = eval(s["r_frame_rate"])
        print(f"  Duration: {dur:.1f}s | {s['width']}x{s['height']} | {s['codec_name']} | {fps:.0f}fps")

In [ ]:
# ═══════════════════════════════════════════
# STEP 4: Dub via ElevenLabs
# ═══════════════════════════════════════════

import httpx
import time

assert ELEVENLABS_API_KEY, "Set ELEVENLABS_API_KEY in the config cell above"

ELEVENLABS_BASE = "https://api.elevenlabs.io/v1"
headers = {"xi-api-key": ELEVENLABS_API_KEY}

# Create dubbing job (send URL so ElevenLabs downloads it)
print(f"Creating ElevenLabs dubbing: {TARGET_LANG}...")
resp = httpx.post(
    f"{ELEVENLABS_BASE}/dubbing",
    headers=headers,
    data={
        "source_url": YOUTUBE_URL,
        "target_lang": TARGET_LANG,
        "mode": "automatic",
        "watermark": "true",
    },
    timeout=300.0,
)
resp.raise_for_status()
dubbing_id = resp.json()["dubbing_id"]
print(f"Dubbing job created: {dubbing_id}")

# Poll until done
print("Waiting for dubbing to complete...")
while True:
    resp = httpx.get(
        f"{ELEVENLABS_BASE}/dubbing/{dubbing_id}",
        headers=headers,
        timeout=30.0,
    )
    resp.raise_for_status()
    status = resp.json()["status"]
    print(f"  Status: {status}")
    if status == "dubbed":
        break
    if status == "failed":
        raise RuntimeError(f"Dubbing failed: {resp.json()}")
    time.sleep(10)

print("Dubbing complete!")

In [ ]:
# ═══════════════════════════════════════════
# STEP 5: Download dubbed audio + convert to WAV
# ═══════════════════════════════════════════

import tempfile

# Download dubbed content
print(f"Downloading dubbed audio ({TARGET_LANG})...")
resp = httpx.get(
    f"{ELEVENLABS_BASE}/dubbing/{dubbing_id}/audio/{TARGET_LANG}",
    headers=headers,
    timeout=300.0,
)
resp.raise_for_status()

# ElevenLabs returns a video — extract audio
tmp_dubbed = "test_data/dubbed_raw.mp4"
with open(tmp_dubbed, "wb") as f:
    f.write(resp.content)

AUDIO_PATH = "test_data/dubbed_audio.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", tmp_dubbed, "-vn", "-ar", "16000", "-ac", "1", AUDIO_PATH],
    check=True,
    capture_output=True,
)

size_mb = os.path.getsize(AUDIO_PATH) / (1024 * 1024)
print(f"Dubbed audio ready: {AUDIO_PATH} ({size_mb:.1f} MB)")

In [ ]:
# ═══════════════════════════════════════════
# STEP 6: Preview source video + dubbed audio
# ═══════════════════════════════════════════

from IPython.display import Video, Audio, display, HTML

print("Source video:")
display(Video(VIDEO_PATH, embed=True, width=320))

print(f"\nDubbed audio ({TARGET_LANG}):")
display(Audio(AUDIO_PATH))

In [ ]:
# ═══════════════════════════════════════════
# STEP 7: Run MuseTalk lipsync
# ═══════════════════════════════════════════

# Write inference config
config = f"""task_0:
  video_path: "{VIDEO_PATH}"
  audio_path: "{AUDIO_PATH}"
"""

os.makedirs("configs/inference", exist_ok=True)
with open("configs/inference/armageddon_test.yaml", "w") as f:
    f.write(config)

print("Running MuseTalk v1.5 lipsync...")
print("(This may take a few minutes on T4)\n")

!python -m scripts.inference \
    --inference_config configs/inference/armageddon_test.yaml \
    --result_dir results/armageddon_test \
    --unet_model_path models/musetalkV15/unet.pth \
    --unet_config models/musetalkV15/musetalk.json \
    --version v15

In [ ]:
# ═══════════════════════════════════════════
# STEP 8: View result + download
# ═══════════════════════════════════════════

import glob
from google.colab import files

outputs = glob.glob("results/armageddon_test/**/*.mp4", recursive=True)

if outputs:
    output_file = outputs[0]
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"Lipsync result: {output_file} ({size_mb:.1f} MB)")
    print()

    # Side by side comparison
    display(HTML("<h3>Original</h3>"))
    display(Video(VIDEO_PATH, embed=True, width=320))
    display(HTML(f"<h3>Lipsynced ({TARGET_LANG})</h3>"))
    display(Video(output_file, embed=True, width=320))

    print("\nDownloading result...")
    files.download(output_file)
else:
    print("No output found. Check logs above.")
    !find results/ -type f 2>/dev/null || echo "No results directory"